In [32]:
import json
import math
import os
from typing import List, Any, Dict, Optional

def compute_ndcg_at_k(ranked_doc_ids: List[Any], positive_ids: List[Any], k: int = 10) -> float:
    pos_set = set(positive_ids)
    if not pos_set:
        return 0.0
    dcg = 0.0
    for rank, doc_id in enumerate(ranked_doc_ids[:k], start=1):
        rel = 1.0 if doc_id in pos_set else 0.0
        if rel > 0:
            dcg += rel / math.log2(rank + 1)
    ideal_rels = [1.0] * min(len(pos_set), k)
    idcg = 0.0
    for rank, rel in enumerate(ideal_rels, start=1):
        idcg += rel / math.log2(rank + 1)
    return 0.0 if idcg == 0.0 else dcg / idcg

compute_ndcg_at_k([1, 2, 3, 4, 5,6,7,8,9,11,12,13,14], [3,6])

0.5249810332008933

In [7]:
topk_doc_ids= [
        "2887",
        "14730",
        "90",
        "14722",
        "14731",
        "1197",
        "4768",
        "815",
        "14718",
        "8310",
        "5410",
        "14720",
        "1917",
        "1915",
        "14729",
        "14723",
        "1520",
        "816",
        "1918",
        "1306",
        "8331",
        "7579",
        "1573",
        "14719",
        "8032",
        "4004",
        "3062",
        "3888",
        "4007",
        "14726",
        "4012",
        "3076",
        "808",
        "7974",
        "1916",
        "3999",
        "12135",
        "3212",
        "4771",
        "3581",
        "3590",
        "8048",
        "4009",
        "4003",
        "3211",
        "3444",
        "3450",
        "4008",
        "4006",
        "93",
        "3471",
        "4777",
        "2403",
        "8661",
        "10878",
        "735",
        "8682"]

In [8]:
systemRole = '''
You are an expert experimental scientist judging whether a paragraph satisfies a query's logical constraints for Alternative Retrieval (ALT).
'''

prompt = '''
Your task is to make a strict binary judgment: does the paragraph fully meet the query's requirements?

Here are the constraints you must follow:
(1) Identify the query's core requirements:
    - The qualitative goal that needs to be achieved
    - The explicit restrictions or avoidances (what must not be used)
(2) Check if the paragraph describes an experimental procedure that:
    - Clearly targets or achieves that goal
    - Does not use the restricted items in the actual protocol
    - Provides a plausible alternative strategy consistent with the restrictions

Important guidelines:
- Use only the information in the query and paragraph, with normal scientific synonyms allowed
- Mentioning restricted items only in citations or discussion doesn't count as "using" them
- If restricted items are clearly used or the goal isn't addressed, answer false
- When uncertain, default to false

Input (JSON):
{
  "query_text": Method for reducing graphene oxide to graphene that does not rely on chemical reducing agents,
  "paragraph": {paragraph}
}

Output (JSON only):
{
  "is_relevant": true/false,
  "reasoning": "Brief explanation of the goal and restrictions, and how the paragraph handles them."
}
'''

In [11]:
import json

para_path = '/home/liangwenliang/检索数据集/seed_data/electrochemistry/all_paragraphs.json'
with open(para_path, "r", encoding="utf-8") as f:
    para_data =  json.load(f)
print(len(topk_doc_ids))
para_dic = {d["id"]:d['paragraph'] for d in para_data}
data = [para_dic[i] for i in topk_doc_ids[:50]]

57


In [31]:
topk_doc_ids[40]

'3590'

In [30]:
print(prompt.replace('{paragraph}',data[46]))


Your task is to make a strict binary judgment: does the paragraph fully meet the query's requirements?

Here are the constraints you must follow:
(1) Identify the query's core requirements:
    - The qualitative goal that needs to be achieved
    - The explicit restrictions or avoidances (what must not be used)
(2) Check if the paragraph describes an experimental procedure that:
    - Clearly targets or achieves that goal
    - Does not use the restricted items in the actual protocol
    - Provides a plausible alternative strategy consistent with the restrictions

Important guidelines:
- Use only the information in the query and paragraph, with normal scientific synonyms allowed
- Mentioning restricted items only in citations or discussion doesn't count as "using" them
- If restricted items are clearly used or the goal isn't addressed, answer false
- When uncertain, default to false

Input (JSON):
{
  "query_text": Method for reducing graphene oxide to graphene that does not rely on c

In [21]:
from openai import OpenAI
import json_repair

model = 'deepseek-chat'
client = OpenAI(base_url='https://ai.api.xn--fiqs8s/v1', api_key="sk-j4kc3j3nzSyBDNSBDANNq9AcBAgXIOvVTgoebJthFfjdHPeJ")
res = []
for i in range(50):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": systemRole},
            {"role": "user", "content": prompt.replace('{paragraph}',data[i])}
        ],
        temperature=0,
        # max_tokens=max_tokens,
    )
    ans = response.choices[0].message.content
    item = json_repair.loads(ans)
    res.append(item)
    print(f'检索模型排序为rank={i+1}, LLM判定相关性为：{item["is_relevant"]}')
    # break

检索模型排序为rank=1, LLM判定相关性为：False
检索模型排序为rank=2, LLM判定相关性为：False
检索模型排序为rank=3, LLM判定相关性为：True
检索模型排序为rank=4, LLM判定相关性为：False
检索模型排序为rank=5, LLM判定相关性为：False
检索模型排序为rank=6, LLM判定相关性为：True
检索模型排序为rank=7, LLM判定相关性为：False
检索模型排序为rank=8, LLM判定相关性为：False
检索模型排序为rank=9, LLM判定相关性为：False
检索模型排序为rank=10, LLM判定相关性为：False
检索模型排序为rank=11, LLM判定相关性为：False
检索模型排序为rank=12, LLM判定相关性为：False
检索模型排序为rank=13, LLM判定相关性为：False
检索模型排序为rank=14, LLM判定相关性为：False
检索模型排序为rank=15, LLM判定相关性为：False
检索模型排序为rank=16, LLM判定相关性为：False
检索模型排序为rank=17, LLM判定相关性为：False
检索模型排序为rank=18, LLM判定相关性为：False
检索模型排序为rank=19, LLM判定相关性为：False
检索模型排序为rank=20, LLM判定相关性为：False
检索模型排序为rank=21, LLM判定相关性为：False
检索模型排序为rank=22, LLM判定相关性为：False
检索模型排序为rank=23, LLM判定相关性为：True
检索模型排序为rank=24, LLM判定相关性为：False
检索模型排序为rank=25, LLM判定相关性为：False
检索模型排序为rank=26, LLM判定相关性为：False
检索模型排序为rank=27, LLM判定相关性为：False
检索模型排序为rank=28, LLM判定相关性为：False
检索模型排序为rank=29, LLM判定相关性为：False
检索模型排序为rank=30, LLM判定相关性为：True
检索模型排序为rank=31, LLM判定相关性为：False
检索模型排序为rank=32, LLM判定

In [17]:
res

[{'is_relevant': False,
  'reasoning': 'Goal: reduce graphene oxide to graphene without using chemical reducing agents. Restriction: no chemical reductants. Paragraph: describes reduction using zinc powder and concentrated HCl (a metal/acid redox chemical route) and washes to remove metal ions. This explicitly uses chemical reducing agents, so it does not meet the restriction and therefore fails to satisfy the query.'},
 {'is_relevant': False,
  'reasoning': "Goal: provide a method to reduce graphene oxide to graphene without using chemical reducing agents. Restriction: no chemical reductants. The paragraph reports reduction of graphene oxide using zinc as a 'green reductant' (and also mentions KMnO4 for oxidation). Zinc is a chemical reducing agent, so the protocol violates the restriction and does not offer a non-chemical alternative. Therefore it does not meet the query."},
 {'is_relevant': False,
  'reasoning': 'Goal: reduce graphene oxide to graphene without using chemical reducin

In [1]:
import os
import json
import pandas as pd

domains = [
    "biochemical",
    "immunological",
    "electrochemistry",
    "organic_chemistry",
    "nanoparticle",
    "solution_based_inorganic",
]

base_dir = "/home/liangwenliang/检索数据集/data"

keep_cols = ["schema_type", "query", "paragraph"]

for domain in domains:
    input_path = os.path.join(base_dir, domain, "eval_llm_query", "query.json")
    if not os.path.exists(input_path):
        print(f"Skip (not found): {input_path}")
        continue

    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    df = pd.DataFrame(data)
    df = df[keep_cols]

    out_path = os.path.join(base_dir, domain, "eval_llm_query", f"{domain}_query.xlsx")
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="tidy", index=False)

    print(f"Saved: {out_path}  (rows={len(df)})")


Saved: /home/liangwenliang/检索数据集/data/biochemical/eval_llm_query/query.xlsx  (rows=373)
Saved: /home/liangwenliang/检索数据集/data/immunological/eval_llm_query/query.xlsx  (rows=324)
Saved: /home/liangwenliang/检索数据集/data/electrochemistry/eval_llm_query/query.xlsx  (rows=300)
Saved: /home/liangwenliang/检索数据集/data/organic_chemistry/eval_llm_query/query.xlsx  (rows=300)
Saved: /home/liangwenliang/检索数据集/data/nanoparticle/eval_llm_query/query.xlsx  (rows=300)
Saved: /home/liangwenliang/检索数据集/data/solution_based_inorganic/eval_llm_query/query.xlsx  (rows=301)
